---
title: "6 - Redes neuronales con PyTorch"
toc: true
---

::: {.callout-note}
## Atribución
Este apunte es una traducción y leve adaptación de la notebook [`10_neural_nets_with_pytorch.ipynb`](https://github.com/ageron/handson-mlp/blob/main/10_neural_nets_with_pytorch.ipynb) del repositorio [`ageron/handson-mlp`](https://github.com/ageron/handson-mlp), de Aurélien Géron. El material original se distribuye bajo la licencia [Apache License 2.0](https://github.com/ageron/handson-mlp/blob/main/LICENSE). Esta versión explicita que fue traducida y adaptada para este curso.
:::

# Configuración

También requiere Scikit-Learn ≥ 1.6.1:

In [ ]:
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1")

¿Estamos usando Colab o Kaggle?

In [ ]:
# IS_COLAB = "google.colab" in sys.modules
# IS_KAGGLE = "kaggle_secrets" in sys.modules

Si usamos Colab, hay un par de librerías que no vienen preinstaladas, así que tenemos que instalarlas manualmente:

In [ ]:
# if IS_COLAB:
#     %pip install -q optuna torchmetrics

Y por supuesto necesitamos PyTorch, específicamente PyTorch ≥ 2.6.0:

In [ ]:
import torch

assert Version(torch.__version__) >= Version("2.6.0")

Como hicimos en capítulos anteriores, definamos los tamaños de fuente predeterminados para que las figuras se vean mejor:

In [ ]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

# Fundamentos de PyTorch
## Tensores en PyTorch

In [ ]:
import torch

X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

In [ ]:
X.shape

In [ ]:
X.dtype

In [ ]:
X[0, 1]

In [ ]:
X[:, 1]

In [ ]:
10 * (X + 1.0)  # suma y multiplicación elemento a elemento

In [ ]:
X.exp()

In [ ]:
X.mean()

In [ ]:
X.max(dim=0)

In [ ]:
X @ X.T

In [ ]:
import numpy as np

X.numpy()

In [ ]:
torch.tensor(np.array([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]]))

In [ ]:
torch.tensor(np.array([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]]), dtype=torch.float32)

In [ ]:
torch.FloatTensor(np.array([[1.0, 4.0, 7.0], [2.0, 3.0, 6]]))

In [ ]:
# código extra: demostrar torch.from_numpy()
X2_np = np.array([[1.0, 4.0, 7.0], [2.0, 3.0, 6]])
X2 = torch.from_numpy(X2_np)  # X2_np y X2 comparten los mismos datos en memoria
X2_np[0, 1] = 88
X2

In [ ]:
X[:, 1] = -99
X

In [ ]:
X.relu_()
X

Los tensores de PyTorch realmente se parecen a los arrays de NumPy. De hecho, ¡tienen más de 200 funciones en común!

In [ ]:
# código extra: listar funciones que aparecen tanto en NumPy como en PyTorch
functions = lambda mod: set(f for f in dir(mod) if callable(getattr(mod, f)))
", ".join(sorted(functions(torch) & functions(np)))

## Aceleración por hardware

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
M = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
M = M.to(device)
M.device

In [ ]:
M = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=device)

In [ ]:
R = M @ M.T  # ejecutar algunas operaciones en la GPU
R

In [ ]:
M = torch.rand((1000, 1000))  # en la CPU
M @ M.T  # calentamiento
%timeit M @ M.T

M = M.to(device)
M @ M.T  # calentamiento
%timeit M @ M.T

## Autograd

Consideremos una función simple, $f(x) = x^2$.
El cálculo nos dice que la derivada de esta función es $f'(x)=2x$. Evaluemos $f(5)$ y la derivada $f'(5)$ usando autograd. Esperamos encontrar $f(5)=5^2=25$ y $f'(5)=2*5=10$. ¡Veamos!

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
f = x**2
f

In [ ]:
f.backward()
x.grad

In [ ]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad  # paso de gradient descent

In [ ]:
x

Alternativamente, podríamos haber usado este código para el paso de gradient descent (pero usar `no_grad()` es más habitual para esto):

In [ ]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

In [ ]:
x.grad.zero_()

Juntemos todo para obtener nuestro loop de entrenamiento:

In [ ]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x**2  # forward pass
    f.backward()  # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad  # paso de gradient descent
    x.grad.zero_()  # reiniciar los gradientes

La variable `x` es empujada hacia 0, ya que ese es el valor que minimiza $f(x) = x^2$:

In [ ]:
x

# Implementación de regresión lineal
## Regresión lineal usando tensores y Autograd

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42
)

In [ ]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

PyTorch espera que los targets tengan una fila por muestra, así que reorganicemos los targets para que sean vectores columna:

In [ ]:
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_valid = torch.FloatTensor(y_valid).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

In [ ]:
torch.manual_seed(42)
n_features = X_train.shape[1]  # hay 8 features de entrada
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

**Nota**: en la siguiente sección construiremos un modelo casi idéntico usando la API de alto nivel de PyTorch. Sus resultados serán ligeramente distintos porque usará un método diferente de inicialización de parámetros: utilizará una distribución aleatoria uniforme desde $-\frac{1}{2\sqrt 2}$ hasta $+\frac{1}{2\sqrt 2}$ para inicializar tanto los pesos como el término de sesgo. Si querés obtener exactamente el mismo resultado aquí que en la siguiente sección, podés descomentar y ejecutar el código de inicialización de la siguiente celda, en lugar del código de la celda anterior:

In [ ]:
# torch.manual_seed(42)
# n_features = X_train.shape[1]  # hay 8 features de entrada
# r = 2 ** -1.5  # esto es igual a 1 / 2√2
# w = torch.empty(n_features, 1).uniform_(-r, r)
# b = torch.empty(1).uniform_(-r, r)
# w.requires_grad_(True)
# b.requires_grad_(True)

In [ ]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Época {epoch + 1}/{n_epochs}, Pérdida: {loss.item()}")

In [ ]:
X_new = X_test[:3]  # supongamos que estas son nuevas instancias
with torch.no_grad():
    y_pred = X_new @ w + b  # usar los parámetros entrenados para hacer predicciones

In [ ]:
y_pred

## Regresión lineal usando la API de alto nivel de PyTorch

In [ ]:
import torch.nn as nn

torch.manual_seed(42)  # para obtener resultados reproducibles
model = nn.Linear(in_features=n_features, out_features=1)

In [ ]:
model.bias

In [ ]:
model.weight

In [ ]:
for param in model.parameters():
    print(param)

In [ ]:
model(X_train[:2])

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [ ]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Época {epoch + 1}/{n_epochs}, Pérdida: {loss.item()}")

In [ ]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

In [ ]:
X_new = X_test[:3]  # supongamos que estas son nuevas instancias
with torch.no_grad():
    y_pred = model(X_new)  # usar el modelo entrenado para hacer predicciones

y_pred

# Implementación de un MLP de regresión

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50), nn.ReLU(), nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1)
)

In [ ]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

# Implementación de mini-batch gradient descent usando DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
# código extra – construir el modelo igual que antes
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50), nn.ReLU(), nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1)
)

model = model.to(device)

# código extra – construir el optimizador y la función de pérdida, como antes
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()

In [ ]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Época {epoch + 1}/{n_epochs}, Pérdida: {mean_loss:.4f}")

In [ ]:
train(model, optimizer, mse, train_loader, n_epochs)

# Evaluación del modelo

In [ ]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

In [ ]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

In [ ]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()


evaluate(model, valid_loader, rmse)

In [ ]:
valid_mse.sqrt()

In [ ]:
evaluate(
    model,
    valid_loader,
    mse,
    aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)),
)

In [ ]:
import torchmetrics


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reiniciar la métrica al comienzo
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # actualizarla en cada iteración
    return metric.compute()  # calcular el resultado final al final

In [ ]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

In [ ]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate_tm(model, valid_loader, metric).item())
        print(
            f"Época {epoch + 1}/{n_epochs}, "
            f"pérdida de entrenamiento: {history['train_losses'][-1]:.4f}, "
            f"métrica de entrenamiento: {history['train_metrics'][-1]:.4f}, "
            f"métrica de validación: {history['valid_metrics'][-1]:.4f}"
        )
    return history

In [ ]:
torch.manual_seed(42)
learning_rate = 0.01
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 30),
    nn.ReLU(),
    nn.Linear(30, 1),
)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train2(model, optimizer, mse, rmse, train_loader, valid_loader, n_epochs)

# Como calculamos la métrica de entrenamiento
plt.plot(np.arange(n_epochs) + 0.5, history["train_metrics"], ".--", label="Entrenamiento")
plt.plot(np.arange(n_epochs) + 1.0, history["valid_metrics"], ".-", label="Validación")
plt.xlabel("Época")
plt.ylabel("RMSE")
plt.grid()
plt.title("Curvas de aprendizaje")
plt.axis([0.5, 20, 0.4, 1.0])
plt.legend()
plt.show()

# Construcción de modelos no secuenciales usando módulos personalizados

In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50),
            nn.ReLU(),
            nn.Linear(50, 40),
            nn.ReLU(),
            nn.Linear(40, 30),
            nn.ReLU(),
        )
        self.output_layer = nn.Linear(30 + n_features, 1)

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [ ]:
torch.manual_seed(42)
model = WideAndDeep(n_features).to(device)
learning_rate = 0.002  # el modelo cambió, así que también cambió el learning rate óptimo

In [ ]:
# código extra: entrenar el modelo, exactamente como nuestros modelos anteriores
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train2(model, optimizer, mse, rmse, train_loader, valid_loader, n_epochs)

In [ ]:
class WideAndDeepV2(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50),
            nn.ReLU(),
            nn.Linear(50, 40),
            nn.ReLU(),
            nn.Linear(40, 30),
            nn.ReLU(),
        )
        self.output_layer = nn.Linear(30 + 5, 1)

    def forward(self, X):
        X_wide = X[:, :5]
        X_deep = X[:, 2:]
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [ ]:
torch.manual_seed(42)
model = WideAndDeepV2(n_features).to(device)

In [ ]:
# código extra: entrenar el modelo, exactamente como nuestros modelos anteriores
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train2(model, optimizer, mse, rmse, train_loader, valid_loader, n_epochs)

## Construcción de modelos con múltiples entradas

In [ ]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50),
            nn.ReLU(),
            nn.Linear(50, 40),
            nn.ReLU(),
            nn.Linear(40, 30),
            nn.ReLU(),
        )
        self.output_layer = nn.Linear(30 + 5, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [ ]:
torch.manual_seed(42)
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
valid_loader_wd = DataLoader(valid_data_wd, batch_size=32)
test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32)

In [ ]:
def evaluate_multi_in(model, data_loader, metric):
    model.eval()
    metric.reset()  # reiniciar la métrica al comienzo
    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch_wide, X_batch_deep)
            metric.update(y_pred, y_batch)  # actualizarla en cada iteración
    return metric.compute()  # calcular el resultado final al final


def train_multi_in(
    model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs
):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        for *X_batch_inputs, y_batch in train_loader:
            model.train()
            X_batch_inputs = [X.to(device) for X in X_batch_inputs]
            y_batch = y_batch.to(device)
            y_pred = model(*X_batch_inputs)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_multi_in(model, valid_loader, metric).item()
        )
        print(
            f"Época {epoch + 1}/{n_epochs}, "
            f"pérdida de entrenamiento: {history['train_losses'][-1]:.4f}, "
            f"métrica de entrenamiento: {history['train_metrics'][-1]:.4f}, "
            f"métrica de validación: {history['valid_metrics'][-1]:.4f}"
        )
    return history


torch.manual_seed(42)
learning_rate = 0.01
model = WideAndDeepV3(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train_multi_in(
    model, optimizer, mse, rmse, train_loader_wd, valid_loader_wd, n_epochs
)

In [ ]:
class WideAndDeepDataset(torch.utils.data.Dataset):
    def __init__(self, X_wide, X_deep, y):
        self.X_wide = X_wide
        self.X_deep = X_deep
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        input_dict = {"X_wide": self.X_wide[idx], "X_deep": self.X_deep[idx]}
        return input_dict, self.y[idx]

In [ ]:
torch.manual_seed(42)
train_data_named = WideAndDeepDataset(
    X_wide=X_train[:, :5], X_deep=X_train[:, 2:], y=y_train
)
train_loader_named = DataLoader(train_data_named, batch_size=32, shuffle=True)
valid_data_named = WideAndDeepDataset(
    X_wide=X_valid[:, :5], X_deep=X_valid[:, 2:], y=y_valid
)
valid_loader_named = DataLoader(valid_data_named, batch_size=32)
test_data_named = WideAndDeepDataset(
    X_wide=X_test[:, :5], X_deep=X_test[:, 2:], y=y_test
)
test_loader_named = DataLoader(test_data_named, batch_size=32)

In [ ]:
def evaluate_named(model, data_loader, metric):
    model.eval()
    metric.reset()  # reiniciar la métrica al comienzo
    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred = model(X_wide=inputs["X_wide"], X_deep=inputs["X_deep"])
            metric.update(y_pred, y_batch)
    return metric.compute()  # calcular el resultado final al final


def train_named(
    model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs
):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        for inputs, y_batch in train_loader:
            model.train()
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred = model(**inputs)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_named(model, valid_loader, metric).item()
        )
        print(
            f"Época {epoch + 1}/{n_epochs}, "
            f"pérdida de entrenamiento: {history['train_losses'][-1]:.4f}, "
            f"métrica de entrenamiento: {history['train_metrics'][-1]:.4f}, "
            f"métrica de validación: {history['valid_metrics'][-1]:.4f}"
        )
    return history


torch.manual_seed(42)
learning_rate = 0.01
model = WideAndDeepV3(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train_named(
    model, optimizer, mse, rmse, train_loader_named, valid_loader_named, n_epochs
)

## Construcción de modelos con múltiples salidas

In [ ]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50),
            nn.ReLU(),
            nn.Linear(50, 40),
            nn.ReLU(),
            nn.Linear(40, 30),
            nn.ReLU(),
        )
        self.output_layer = nn.Linear(30 + 5, 1)
        self.aux_output_layer = nn.Linear(30, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        return main_output, aux_output

In [ ]:
import torchmetrics


def evaluate_multi_out(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred, _ = model(**inputs)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train_multi_out(
    model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs
):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        for inputs, y_batch in train_loader:
            model.train()
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred, y_pred_aux = model(**inputs)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_multi_out(model, valid_loader, metric).item()
        )
        print(
            f"Época {epoch + 1}/{n_epochs}, "
            f"pérdida de entrenamiento: {history['train_losses'][-1]:.4f}, "
            f"métrica de entrenamiento: {history['train_metrics'][-1]:.4f}, "
            f"métrica de validación: {history['valid_metrics'][-1]:.4f}"
        )
    return history


torch.manual_seed(42)
learning_rate = 0.01
model = WideAndDeepV4(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train_multi_out(
    model, optimizer, mse, rmse, train_loader_named, valid_loader_named, n_epochs
)

# Construcción de un clasificador de imágenes con PyTorch

## Uso de TorchVision para cargar el dataset

In [ ]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000]
)

In [ ]:
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

Cada entrada es una tupla `(image, target)`:

In [ ]:
X_sample, y_sample = train_data[0]

Cada imagen tiene la forma \[canales, filas, columnas\]. Las imágenes en escala de grises como las de Fashion MNIST tienen un solo canal (mientras que las imágenes RGB tienen 3, y otros tipos de imágenes, como las satelitales, pueden tener muchos más). Las imágenes de Fashion son en escala de grises y de 28x28 píxeles:

In [ ]:
X_sample.shape

In [ ]:
X_sample.dtype

In [ ]:
train_and_valid_data.classes[y_sample]

## Construcción del clasificador

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(
    n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10
).to(device)
xentropy = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
_ = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs)

In [ ]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred = y_pred_logits.argmax(dim=1)  # índice del logit más grande
y_pred

In [ ]:
[train_and_valid_data.classes[index] for index in y_pred]

Verifiquemos si el modelo hizo las predicciones correctas:

In [ ]:
y_new[:3]

¡Todas correctas! 😃

In [ ]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)
if device == "mps":
    y_proba = y_proba.cpu()
y_proba.round(decimals=3)

In [ ]:
y_top4_values, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_values, dim=1)
if device == "mps":
    y_top4_probas = y_top4_probas.cpu()
y_top4_probas.round(decimals=3)

In [ ]:
y_top4_indices

In [ ]:
sum([param.numel() for param in model.parameters()])

# Ajuste de hiperparámetros usando Optuna

In [ ]:
import optuna


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(
        n_inputs=1 * 28 * 28, n_hidden1=n_hidden, n_hidden2=n_hidden, n_classes=10
    ).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    history = train2(
        model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs=10
    )
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy

In [ ]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

In [ ]:
study.best_params

In [ ]:
study.best_value

In [ ]:
def objective(trial, train_loader, valid_loader):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(
        n_inputs=1 * 28 * 28, n_hidden1=n_hidden, n_hidden2=n_hidden, n_classes=10
    ).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(
            model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs=1
        )
        validation_accuracy = max(history["valid_metrics"])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

In [ ]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader
)

In [ ]:
from functools import partial

objective_with_data = partial(
    objective, train_loader=train_loader, valid_loader=valid_loader
)

In [ ]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective_with_data, n_trials=20)

In [ ]:
study.best_value

In [ ]:
study.best_params

# Guardar y cargar un modelo de PyTorch

In [ ]:
torch.save(model, "my_fashion_mnist.pt")

In [ ]:
loaded_model = torch.load("my_fashion_mnist.pt", weights_only=False)

In [ ]:
loaded_model.eval()
y_pred_logits = loaded_model(X_new)

In [ ]:
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")

In [ ]:
type(model.state_dict())

In [ ]:
new_model = ImageClassifier(
    n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10
)
loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)
new_model.load_state_dict(loaded_weights)
new_model.eval()

In [ ]:
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {
        "n_inputs": 1 * 28 * 28,
        "n_hidden1": 300,
        "n_hidden2": 100,
        "n_classes": 10,
    },
}
torch.save(model_data, "my_fashion_mnist_model.pt")

In [ ]:
loaded_data = torch.load("my_fashion_mnist_model.pt", weights_only=True)
new_model = ImageClassifier(**loaded_data["model_hyperparameters"])
new_model.load_state_dict(loaded_data["model_state_dict"])
new_model.eval()

# Compilación y optimización de un modelo de PyTorch

In [ ]:
torchscript_model = torch.jit.trace(model, X_new)

In [ ]:
torchscript_model = torch.jit.script(model)

In [ ]:
optimized_model = torch.jit.optimize_for_inference(torchscript_model)

In [ ]:
optimized_model.save("my_fashion_mnist_torchscript.pt")

In [ ]:
loaded_torchscript_model = torch.jit.load("my_fashion_mnist_torchscript.pt")

In [ ]:
y_pred_logits = loaded_torchscript_model(X_new)
y_pred_logits

In [ ]:
compiled_model = torch.compile(model)

In [ ]:
if device == "cuda":
    y_pred_logits = compiled_model(X_new)